In [3]:
import pandas as pd
import os

# AWS CSV 폴더 경로
aws_folder = "/Users/doyoung-gil/Desktop/연구실/기상데이터/AWS"  # ← 여기를 AWS CSV 폴더 경로로 변경

for file in os.listdir(aws_folder):
    if file.lower().endswith(".csv"):
        file_path = os.path.join(aws_folder, file)

        # CSV 불러오기
        df = pd.read_csv(file_path, encoding="utf-8-sig")

        # 컬럼명 변경 (있을 경우만)
        if "최대 순간 풍속(m/s)" in df.columns:
            df.rename(columns={"최대 순간 풍속(m/s)": "최대 풍속(m/s)"}, inplace=True)
            df.to_csv(file_path, index=False, encoding="utf-8-sig")
            print(f"✅ {file} → 컬럼명 변경 완료")
        else:
            print(f"⚠ {file} → 변경할 컬럼 없음")

print("모든 파일 처리 완료!")


✅ 2021WeatherData.csv → 컬럼명 변경 완료
✅ 2020WeatherData.csv → 컬럼명 변경 완료
✅ 2022WeatherData.csv → 컬럼명 변경 완료
✅ 2024WeatherData.csv → 컬럼명 변경 완료
✅ 2023WeatherData.csv → 컬럼명 변경 완료
✅ 2017WeatherData.csv → 컬럼명 변경 완료
✅ 2018WeatherData.csv → 컬럼명 변경 완료
✅ 2019WeatherData.csv → 컬럼명 변경 완료
✅ 2011WeatherData.csv → 컬럼명 변경 완료
✅ 2016WeatherData.csv → 컬럼명 변경 완료
✅ 2014WeatherData.csv → 컬럼명 변경 완료
✅ 2013WeatherData.csv → 컬럼명 변경 완료
✅ 2012WeatherData.csv → 컬럼명 변경 완료
✅ 2015WeatherData.csv → 컬럼명 변경 완료
모든 파일 처리 완료!


In [5]:
import os
import re

# 폴더 경로
asos_folder = "/Users/doyoung-gil/Desktop/연구실/기상데이터/ASOS"
aws_folder = "/Users/doyoung-gil/Desktop/연구실/기상데이터/AWS"

def rename_files(folder, prefix):
    for filename in os.listdir(folder):
        # 연도 추출 (파일명 안에 4자리 숫자 찾기)
        match = re.search(r"(20\d{2})", filename)
        if match:
            year = match.group(1)
            new_name = f"{prefix}_{year}.csv"
            old_path = os.path.join(folder, filename)
            new_path = os.path.join(folder, new_name)
            os.rename(old_path, new_path)
            print(f"{filename} → {new_name}")
        else:
            print(f"❌ 연도 못 찾음: {filename}")

# ASOS와 AWS 각각 실행
rename_files(asos_folder, "ASOS")
rename_files(aws_folder, "AWS")

print("✅ 파일명 변경 완료!")


2023_WeatherData.csv → ASOS_2023.csv
2011_WeatherData.csv → ASOS_2011.csv
2016_WeatherData.csv → ASOS_2016.csv
2024_WeatherData.csv → ASOS_2024.csv
2019_WeatherData.csv → ASOS_2019.csv
2017_WeatherData.csv → ASOS_2017.csv
2018_WeatherData.csv → ASOS_2018.csv
2022_WeatherData.csv → ASOS_2022.csv
2015_WeatherData.csv → ASOS_2015.csv
2020_WeatherData.csv → ASOS_2020.csv
2012_WeatherData.csv → ASOS_2012.csv
2021_WeatherData.csv → ASOS_2021.csv
2013_WeatherData.csv → ASOS_2013.csv
2014_WeatherData.csv → ASOS_2014.csv
2021WeatherData.csv → AWS_2021.csv
❌ 연도 못 찾음: .DS_Store
2020WeatherData.csv → AWS_2020.csv
2022WeatherData.csv → AWS_2022.csv
2024WeatherData.csv → AWS_2024.csv
2023WeatherData.csv → AWS_2023.csv
2017WeatherData.csv → AWS_2017.csv
2018WeatherData.csv → AWS_2018.csv
2019WeatherData.csv → AWS_2019.csv
2011WeatherData.csv → AWS_2011.csv
2016WeatherData.csv → AWS_2016.csv
2014WeatherData.csv → AWS_2014.csv
2013WeatherData.csv → AWS_2013.csv
2012WeatherData.csv → AWS_2012.csv
2015We

### 1997-2006, 2007-2010년 자료 년도별로 쪼개기

In [8]:
from pathlib import Path
import pandas as pd

IN = Path("/Users/doyoung-gil/Desktop/연구실/데이터/AWS/AWS_1997-2006.csv")
OUT_DIR = Path("/Users/doyoung-gil/Desktop/연구실/데이터/AWS")
OUT_DIR.mkdir(exist_ok=True)

# 인코딩 자동 시도
try:
    df = pd.read_csv(IN, encoding="utf-8-sig")
except UnicodeDecodeError:
    df = pd.read_csv(IN, encoding="cp949")

yr = pd.to_datetime(df["일시"], errors="coerce").dt.year
for y in range(1997, 2007):
    sub = df[yr == y]
    if not sub.empty:
        sub.to_csv(OUT_DIR / f"ASOS_{y}.csv", index=False, encoding="utf-8-sig")

print("저장 위치:", OUT_DIR.resolve())


저장 위치: /Users/doyoung-gil/Desktop/연구실/데이터/AWS


### ASOS+AWS 병합

In [12]:
import pandas as pd
import os

# 폴더 경로
asos_folder = "/Users/doyoung-gil/Desktop/연구실/데이터/ASOS"  # ASOS 연도별 파일
aws_folder = "/Users/doyoung-gil/Desktop/연구실/데이터/AWS"    # AWS 연도별 파일
output_folder = "/Users/doyoung-gil/Desktop/연구실/데이터/ASOS+AWS"  # 저장 폴더

# 연도 범위 (2024 → 2011)
years = range(2010, 1996, -1)

for year in years:
    asos_path = os.path.join(asos_folder, f"ASOS_{year}.csv")
    aws_path = os.path.join(aws_folder, f"AWS_{year}.csv")

    if not os.path.exists(asos_path):
        print(f"{year}년 ASOS 데이터 없음")
        continue
    if not os.path.exists(aws_path):
        print(f"{year}년 AWS 데이터 없음")
        continue

    # 파일 불러오기
    asos_df = pd.read_csv(asos_path, encoding="utf-8-sig")
    aws_df = pd.read_csv(aws_path, encoding="utf-8-sig")

    # 공통 컬럼만 추출
    common_cols = list(set(asos_df.columns) & set(aws_df.columns))
    asos_df = asos_df[common_cols]
    aws_df = aws_df[common_cols]

    # 합치기
    combined_df = pd.concat([asos_df, aws_df], ignore_index=True)

    # 중복 제거
    if "지점" in combined_df.columns and "일시" in combined_df.columns:
        combined_df.drop_duplicates(subset=["지점", "일시"], inplace=True)
        combined_df["일시"] = pd.to_datetime(combined_df["일시"], errors="coerce")

        # 지점 번호, 일시 순 정렬
        combined_df = combined_df.sort_values(by=["지점", "일시"], ascending=[True, True])
        
        desired_order = [
            "지점", "지점명", "일시",
            "평균기온(°C)", "최저기온(°C)", "최고기온(°C)",
            "일강수량(mm)", "최대 풍속(m/s)", "평균 풍속(m/s)"
        ]
        front = [c for c in desired_order if c in combined_df.columns]
        rest = [c for c in combined_df.columns if c not in front]
        combined_df = combined_df[front + rest]

    # 저장
    save_path = os.path.join(output_folder, f"ASOS_AWS_{year}.csv")
    combined_df.to_csv(save_path, index=False, encoding="utf-8-sig")

    print(f"{year}년 합치기 완료 → {save_path}")

print("✅ 모든 연도 처리 완료!")


2010년 합치기 완료 → /Users/doyoung-gil/Desktop/연구실/데이터/ASOS+AWS/ASOS_AWS_2010.csv
2009년 합치기 완료 → /Users/doyoung-gil/Desktop/연구실/데이터/ASOS+AWS/ASOS_AWS_2009.csv
2008년 합치기 완료 → /Users/doyoung-gil/Desktop/연구실/데이터/ASOS+AWS/ASOS_AWS_2008.csv
2007년 합치기 완료 → /Users/doyoung-gil/Desktop/연구실/데이터/ASOS+AWS/ASOS_AWS_2007.csv
2006년 합치기 완료 → /Users/doyoung-gil/Desktop/연구실/데이터/ASOS+AWS/ASOS_AWS_2006.csv
2005년 합치기 완료 → /Users/doyoung-gil/Desktop/연구실/데이터/ASOS+AWS/ASOS_AWS_2005.csv
2004년 합치기 완료 → /Users/doyoung-gil/Desktop/연구실/데이터/ASOS+AWS/ASOS_AWS_2004.csv
2003년 합치기 완료 → /Users/doyoung-gil/Desktop/연구실/데이터/ASOS+AWS/ASOS_AWS_2003.csv
2002년 합치기 완료 → /Users/doyoung-gil/Desktop/연구실/데이터/ASOS+AWS/ASOS_AWS_2002.csv
2001년 합치기 완료 → /Users/doyoung-gil/Desktop/연구실/데이터/ASOS+AWS/ASOS_AWS_2001.csv
2000년 합치기 완료 → /Users/doyoung-gil/Desktop/연구실/데이터/ASOS+AWS/ASOS_AWS_2000.csv
1999년 합치기 완료 → /Users/doyoung-gil/Desktop/연구실/데이터/ASOS+AW

### ASOS+AWS파일에 메타데이터 추가

In [13]:
import pandas as pd
import os, glob

# ========= 경로 설정 =========
ASOS_DIR   = "/Users/doyoung-gil/Desktop/연구실/데이터/ASOS"        # ASOS_YYYY.csv 있는 폴더
AWS_DIR    = "/Users/doyoung-gil/Desktop/연구실/데이터/AWS"         # AWS_YYYY.csv 있는 폴더
OUT_DIR    = "/Users/doyoung-gil/Desktop/연구실/데이터/ASOS+AWS"    # 결과 저장 폴더
META_FILES = [
    "/Users/doyoung-gil/Desktop/연구실/데이터/관측소 메타데이터/META_관측지점정보.csv",
    "/Users/doyoung-gil/Desktop/연구실/데이터/관측소 메타데이터/META_관측지점정보2.csv",
]
YEARS = range(2010, 1997-1, -1)   # 2010 → 1997
os.makedirs(OUT_DIR, exist_ok=True)

def read_csv_smart(path):
    """인코딩 자동 시도로 CSV 읽기"""
    for enc in ("utf-8-sig", "euc-kr", "cp949", "utf-8"):
        try:
            return pd.read_csv(path, encoding=enc)
        except Exception:
            pass
    return pd.read_csv(path)

# ========= 1) 메타(관측지점) 결합/정리 =========
meta_list = [read_csv_smart(p) for p in META_FILES]
meta = pd.concat(meta_list, ignore_index=True)

# 컬럼 표준화
ren = {}
if "지점번호" in meta.columns and "지점" not in meta.columns:
    ren["지점번호"] = "지점"
if "위도(°)" in meta.columns and "위도" not in meta.columns:
    ren["위도(°)"] = "위도"
if "경도(°)" in meta.columns and "경도" not in meta.columns:
    ren["경도(°)"] = "경도"
meta = meta.rename(columns=ren)

# 지점 코드 숫자화
if "지점" in meta.columns:
    meta["지점"] = pd.to_numeric(meta["지점"], errors="coerce")

# 종료일 기준으로 지점별 최신 1행만 남기기(운영 중 우선)
if "종료일" in meta.columns:
    m = meta.copy()
    m["종료일_ts"] = pd.to_datetime(m["종료일"], errors="coerce")
    # 운영 중(NaT)이 앞으로 오도록 정렬키 구성
    m["종료됨"] = m["종료일_ts"].notna()          # False(운영중) < True(종료됨)
    m = m.sort_values(["지점", "종료됨", "종료일_ts"])
    meta_keep = m.groupby("지점", as_index=False).first()
else:
    meta_keep = meta.drop_duplicates(subset=["지점"], keep="last")

# 병합에 쓸 컬럼만 남김(지점명은 원본 파일의 것을 유지)
meta_keep = meta_keep[[c for c in ["지점", "위도", "경도"] if c in meta_keep.columns]].copy()

# ========= 2) 연도별 ASOS+AWS 병합 + 위경도 삽입 =========
for y in YEARS:
    asos_path = os.path.join(ASOS_DIR, f"ASOS_{y}.csv")
    aws_path  = os.path.join(AWS_DIR,  f"AWS_{y}.csv")

    if not os.path.exists(asos_path) or not os.path.exists(aws_path):
        print(f"{y}: ASOS 또는 AWS 파일 없음 → 건너뜀")
        continue

    asos = read_csv_smart(asos_path)
    aws  = read_csv_smart(aws_path)

    # (안전) AWS의 '최대 순간 풍속' 표기를 '최대 풍속'으로 통일
    if "최대 순간 풍속(m/s)" in aws.columns and "최대 풍속(m/s)" not in aws.columns:
        aws = aws.rename(columns={"최대 순간 풍속(m/s)": "최대 풍속(m/s)"})

    # 열 유니온(스키마 맞춤): 없는 열은 NaN으로 생성
    all_cols = sorted(set(asos.columns) | set(aws.columns))
    for col in all_cols:
        if col not in asos.columns: asos[col] = pd.NA
        if col not in aws.columns:  aws[col]  = pd.NA
    asos = asos[all_cols]
    aws  = aws[all_cols]

    # 세로 결합
    df = pd.concat([asos, aws], ignore_index=True)

    # 기본 전처리(날짜/지점)
    if "일시" in df.columns:
        df["일시"] = pd.to_datetime(df["일시"], errors="coerce")
    if "지점" in df.columns:
        df["지점"] = pd.to_numeric(df["지점"], errors="coerce")

    # 중복 제거 & 정렬(지점 → 일시)
    if {"지점", "일시"}.issubset(df.columns):
        df = df.drop_duplicates(subset=["지점", "일시"])
        df = df.sort_values(["지점", "일시"])

    # 위/경도 병합(지점 기준, 지점명 충돌 방지)
    df = df.merge(meta_keep, on="지점", how="left")  # 지점명은 원본 유지 → _x/_y 안 생김

    # 혹시 이전 단계에서 지점명_x/_y가 생겼다면 정리(방어 코드)
    if "지점명_x" in df.columns:
        df = df.rename(columns={"지점명_x": "지점명"})
        df = df.drop(columns=["지점명_y"], errors="ignore")

    # ========= 컬럼 순서 재배치: 지점, 지점명, 위도, 경도, 일시, 나머지 =========
    cols = df.columns.tolist()
    head = ["지점", "지점명", "위도", "경도", "일시"]
    front = [c for c in head if c in cols]
    rest  = [c for c in cols if c not in front]
    df = df[front + rest]

    # 저장(덮어쓰기 or 새 파일명 선택)
    save_path = os.path.join(OUT_DIR, f"ASOS_AWS_{y}.csv")  # 덮어쓰기 원치 않으면 _with_coords 등 suffix 추가
    df.to_csv(save_path, index=False, encoding="utf-8-sig")
    miss = int(df["위도"].isna().sum()) if "위도" in df.columns else 0
    print(f"{y}: 저장 → {save_path} (rows={len(df)}, 위경도 미매칭 {miss})")

print("✅ 모든 연도 처리 완료!")


/var/folders/2j/__kxcrvx5ss949vx7sspc6th0000gn/T/ipykernel_2118/1814776225.py:81: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([asos, aws], ignore_index=True)


2010: 저장 → /Users/doyoung-gil/Desktop/연구실/데이터/ASOS+AWS/ASOS_AWS_2010.csv (rows=206533, 위경도 미매칭 0)


/var/folders/2j/__kxcrvx5ss949vx7sspc6th0000gn/T/ipykernel_2118/1814776225.py:81: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([asos, aws], ignore_index=True)


2009: 저장 → /Users/doyoung-gil/Desktop/연구실/데이터/ASOS+AWS/ASOS_AWS_2009.csv (rows=204767, 위경도 미매칭 0)


/var/folders/2j/__kxcrvx5ss949vx7sspc6th0000gn/T/ipykernel_2118/1814776225.py:81: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([asos, aws], ignore_index=True)


2008: 저장 → /Users/doyoung-gil/Desktop/연구실/데이터/ASOS+AWS/ASOS_AWS_2008.csv (rows=204348, 위경도 미매칭 0)


/var/folders/2j/__kxcrvx5ss949vx7sspc6th0000gn/T/ipykernel_2118/1814776225.py:81: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([asos, aws], ignore_index=True)


2007: 저장 → /Users/doyoung-gil/Desktop/연구실/데이터/ASOS+AWS/ASOS_AWS_2007.csv (rows=201251, 위경도 미매칭 0)
2006: 저장 → /Users/doyoung-gil/Desktop/연구실/데이터/ASOS+AWS/ASOS_AWS_2006.csv (rows=142861, 위경도 미매칭 0)
2005: 저장 → /Users/doyoung-gil/Desktop/연구실/데이터/ASOS+AWS/ASOS_AWS_2005.csv (rows=141993, 위경도 미매칭 0)
2004: 저장 → /Users/doyoung-gil/Desktop/연구실/데이터/ASOS+AWS/ASOS_AWS_2004.csv (rows=141649, 위경도 미매칭 0)
2003: 저장 → /Users/doyoung-gil/Desktop/연구실/데이터/ASOS+AWS/ASOS_AWS_2003.csv (rows=138913, 위경도 미매칭 0)
2002: 저장 → /Users/doyoung-gil/Desktop/연구실/데이터/ASOS+AWS/ASOS_AWS_2002.csv (rows=134654, 위경도 미매칭 0)
2001: 저장 → /Users/doyoung-gil/Desktop/연구실/데이터/ASOS+AWS/ASOS_AWS_2001.csv (rows=124700, 위경도 미매칭 0)
2000: 저장 → /Users/doyoung-gil/Desktop/연구실/데이터/ASOS+AWS/ASOS_AWS_2000.csv (rows=116997, 위경도 미매칭 0)
1999: 저장 → /Users/doyoung-gil/Desktop/연구실/데이터/ASOS+AWS/ASOS_AWS_1999.csv (rows=114968, 위경도 미매칭 0)
1998: 저장 → /Users/doyoung-gil/Desktop/연구실/데이터/ASOS+AWS/ASOS_AWS_1998.csv (rows=112812, 위경도 미매칭 0)
1997: 저장 → /Users/do